# NSL-KDD Data Cleansing

This notebook separates the cleansing workflow into small, inspectable steps and writes the cleaned datasets to:

data-pipeline/output/NSL-KDD

Workflow:

1. Load raw train/test files
2. Normalize text columns
3. Convert numeric columns and detect invalid values
4. Replace infinite values and handle missing rows
5. Remove exact duplicates
6. Validate feature ranges
7. Remove metadata and constant columns
8. Add attack-family and binary attack targets
9. Validate and export the cleaned datasets

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 0. Paths, schema, and cleansing configuration

In [ ]:
RAW_DIR = Path("../dataset/rawdata/NSL-KDD")
OUTPUT_DIR = Path("output/NSL-KDD")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Raw data directory not found: {RAW_DIR.resolve()}")

RAW_FILES = {
    "KDDTrain": "KDDTrain+.txt",
    "KDDTest": "KDDTest+.txt",
    "KDDTest-21": "KDDTest-21.txt",
}

columns = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "label",
    "difficulty",
]

categorical_cols = ["protocol_type", "service", "flag"]
target_col = "label"
numeric_cols = [col for col in columns if col not in categorical_cols + [target_col]]

rate_cols = [
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
]
non_negative_cols = [col for col in numeric_cols if col != "difficulty"]

print(f"Raw data directory: {RAW_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## 1. Attack-family mapping

In [ ]:
attack_family_map = {
    "back": "DoS",
    "land": "DoS",
    "neptune": "DoS",
    "pod": "DoS",
    "smurf": "DoS",
    "teardrop": "DoS",
    "ipsweep": "Probe",
    "nmap": "Probe",
    "portsweep": "Probe",
    "satan": "Probe",
    "ftp_write": "R2L",
    "guess_passwd": "R2L",
    "imap": "R2L",
    "multihop": "R2L",
    "phf": "R2L",
    "spy": "R2L",
    "warezclient": "R2L",
    "warezmaster": "R2L",
    "buffer_overflow": "U2R",
    "loadmodule": "U2R",
    "perl": "U2R",
    "rootkit": "U2R",
}

def add_targets(frame):
    frame = frame.copy()
    frame["attack_family"] = frame["label"].map(attack_family_map)
    normal_mask = frame["attack_family"].isna() & frame["label"].eq("normal")
    frame.loc[normal_mask, "attack_family"] = "Normal"
    frame["attack_family"] = frame["attack_family"].fillna("Unknown")
    frame["is_attack"] = frame["label"].ne("normal").astype("int8")
    return frame

## 2. Load raw datasets

In [ ]:
raw_datasets = {}
raw_summary = []

for dataset_name, filename in RAW_FILES.items():
    raw_path = RAW_DIR / filename
    if not raw_path.exists():
        raise FileNotFoundError(f"Missing raw file: {raw_path.resolve()}")

    raw_frame = pd.read_csv(raw_path, names=columns)
    raw_datasets[dataset_name] = raw_frame
    raw_summary.append({
        "dataset": dataset_name,
        "rows_raw": len(raw_frame),
        "columns_raw": raw_frame.shape[1],
        "missing_raw": int(raw_frame.isna().sum().sum()),
        "duplicates_raw": int(raw_frame.duplicated().sum()),
    })

display(pd.DataFrame(raw_summary))

## 3. Normalize text columns

Trim whitespace and convert categorical values and labels to lowercase. This keeps train/test category names consistent.

In [ ]:
normalized_datasets = {}
normalization_summary = []

for dataset_name, raw_frame in raw_datasets.items():
    frame = raw_frame.copy()

    for col in categorical_cols + [target_col]:
        frame[col] = frame[col].astype("string").str.strip().str.lower()

    normalized_datasets[dataset_name] = frame
    normalization_summary.append({
        "dataset": dataset_name,
        "normalized_columns": ", ".join(categorical_cols + [target_col]),
        "unique_labels": frame[target_col].nunique(),
    })

display(pd.DataFrame(normalization_summary))

## 4. Convert numeric columns

Convert all expected numeric columns with pandas and count values that cannot be converted. Invalid numeric values stop the pipeline so they are not silently hidden.

In [ ]:
numeric_datasets = {}
numeric_summary = []

for dataset_name, normalized_frame in normalized_datasets.items():
    frame = normalized_frame.copy()
    invalid_numeric_values = 0

    for col in numeric_cols:
        original = frame[col]
        converted = pd.to_numeric(original, errors="coerce")
        invalid_numeric_values += int((converted.isna() & original.notna()).sum())
        frame[col] = converted

    infinite_values = int(np.isinf(frame[numeric_cols].to_numpy(dtype=float)).sum())
    numeric_datasets[dataset_name] = frame
    numeric_summary.append({
        "dataset": dataset_name,
        "invalid_numeric_values": invalid_numeric_values,
        "infinite_values": infinite_values,
    })

numeric_summary_df = pd.DataFrame(numeric_summary)
display(numeric_summary_df)

if numeric_summary_df["invalid_numeric_values"].sum() > 0:
    raise ValueError("Invalid numeric values were found; inspect numeric_summary_df before continuing.")

## 5. Handle infinite values and missing rows

Infinite values are converted to missing values. Any incomplete rows are dropped and the number of dropped rows is recorded in the final report.

In [ ]:
missing_datasets = {}
missing_summary = []

for dataset_name, numeric_frame in numeric_datasets.items():
    frame = numeric_frame.replace([np.inf, -np.inf], np.nan)
    missing_cells_before = int(frame.isna().sum().sum())
    rows_dropped_missing = int(frame.isna().any(axis=1).sum())

    if rows_dropped_missing:
        frame = frame.dropna().copy()

    missing_datasets[dataset_name] = frame
    missing_summary.append({
        "dataset": dataset_name,
        "missing_cells_before": missing_cells_before,
        "rows_dropped_missing": rows_dropped_missing,
        "missing_cells_after": int(frame.isna().sum().sum()),
    })

missing_summary_df = pd.DataFrame(missing_summary)
display(missing_summary_df)

## 6. Remove exact duplicates

Only exact duplicate rows are removed. Repeated network patterns that are not exactly identical are preserved.

In [ ]:
deduplicated_datasets = {}
duplicate_summary = []

for dataset_name, missing_frame in missing_datasets.items():
    duplicate_rows = int(missing_frame.duplicated().sum())
    frame = missing_frame.drop_duplicates().reset_index(drop=True)

    deduplicated_datasets[dataset_name] = frame
    duplicate_summary.append({
        "dataset": dataset_name,
        "duplicate_rows_before": duplicate_rows,
        "rows_dropped_duplicates": duplicate_rows,
        "duplicates_after": int(frame.duplicated().sum()),
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary)
display(duplicate_summary_df)

## 7. Validate numeric ranges

Rate features must be between 0 and 1. Other numeric connection features must be non-negative. The pipeline raises an error if invalid values are found.

In [ ]:
range_validated_datasets = {}
range_summary = []

for dataset_name, deduplicated_frame in deduplicated_datasets.items():
    frame = deduplicated_frame.copy()
    invalid_ranges = {}

    for col in rate_cols:
        invalid = (~frame[col].between(0, 1)) & frame[col].notna()
        if invalid.any():
            invalid_ranges[col] = int(invalid.sum())

    for col in non_negative_cols:
        invalid = (frame[col] < 0) & frame[col].notna()
        if invalid.any():
            invalid_ranges[col] = invalid_ranges.get(col, 0) + int(invalid.sum())

    invalid_range_values = sum(invalid_ranges.values())
    range_summary.append({
        "dataset": dataset_name,
        "invalid_range_values": invalid_range_values,
        "range_details": json.dumps(invalid_ranges),
    })

    if invalid_range_values:
        raise ValueError(f"{dataset_name} contains invalid ranges: {invalid_ranges}")

    range_validated_datasets[dataset_name] = frame

range_summary_df = pd.DataFrame(range_summary)
display(range_summary_df)

## 8. Remove metadata and constant features

Remove difficulty because it is dataset metadata, not a connection feature. Remove any constant predictor; for this dataset that includes num_outbound_cmds.

In [ ]:
structurally_cleaned_datasets = {}
structure_summary = []

for dataset_name, range_frame in range_validated_datasets.items():
    frame = range_frame.copy()
    rows_before = len(frame)
    columns_before = frame.shape[1]
    dropped_columns = []

    if "difficulty" in frame.columns:
        frame = frame.drop(columns="difficulty")
        dropped_columns.append("difficulty")

    constant_columns = [
        col for col in frame.columns
        if col not in {"label"} and frame[col].nunique(dropna=False) <= 1
    ]
    if constant_columns:
        frame = frame.drop(columns=constant_columns)
        dropped_columns.extend(constant_columns)

    structurally_cleaned_datasets[dataset_name] = frame
    structure_summary.append({
        "dataset": dataset_name,
        "rows_before": rows_before,
        "rows_after": len(frame),
        "columns_before": columns_before,
        "columns_after": frame.shape[1],
        "dropped_columns": ", ".join(dropped_columns),
    })

structure_summary_df = pd.DataFrame(structure_summary)
display(structure_summary_df)

## 9. Add model targets

In [ ]:
cleaned = {}
target_summary = []

for dataset_name, structural_frame in structurally_cleaned_datasets.items():
    frame = add_targets(structural_frame)
    cleaned[dataset_name] = frame

    target_summary.append({
        "dataset": dataset_name,
        "rows": len(frame),
        "normal_rows": int((frame["is_attack"] == 0).sum()),
        "attack_rows": int((frame["is_attack"] == 1).sum()),
        "unknown_attack_labels": int((frame["attack_family"] == "Unknown").sum()),
    })

display(pd.DataFrame(target_summary))
display(cleaned["KDDTrain"].head())

## 10. Final validation

In [ ]:
final_summary = []

for dataset_name, frame in cleaned.items():
    final_summary.append({
        "dataset": dataset_name,
        "rows_final": len(frame),
        "columns_final": frame.shape[1],
        "missing_final": int(frame.isna().sum().sum()),
        "duplicates_final": int(frame.duplicated().sum()),
        "has_difficulty": "difficulty" in frame.columns,
        "has_constant_num_outbound_cmds": "num_outbound_cmds" in frame.columns,
    })

final_summary_df = pd.DataFrame(final_summary)
display(final_summary_df)

if (
    final_summary_df["missing_final"].sum() > 0
    or final_summary_df["duplicates_final"].sum() > 0
    or final_summary_df["has_difficulty"].any()
    or final_summary_df["has_constant_num_outbound_cmds"].any()
):
    raise ValueError("Final validation failed; inspect final_summary_df.")

## 11. Export cleaned datasets and reports

In [ ]:
report_df = (
    pd.DataFrame(raw_summary)
    .merge(numeric_summary_df, on="dataset")
    .merge(missing_summary_df, on="dataset")
    .merge(duplicate_summary_df, on="dataset")
    .merge(range_summary_df, on="dataset")
    .merge(structure_summary_df, on="dataset")
    .merge(final_summary_df, on="dataset")
)

for dataset_name, frame in cleaned.items():
    output_path = OUTPUT_DIR / f"{dataset_name}_clean.csv"
    frame.to_csv(output_path, index=False)
    print(f"Saved {output_path.resolve()}")

report_df.to_csv(OUTPUT_DIR / "cleaning_report.csv", index=False)

schema_df = pd.DataFrame({
    "column": cleaned["KDDTrain"].columns,
    "dtype": [str(dtype) for dtype in cleaned["KDDTrain"].dtypes],
})
schema_df.to_csv(OUTPUT_DIR / "cleaned_schema.csv", index=False)

config = {
    "raw_directory": str(RAW_DIR.resolve()),
    "output_directory": str(OUTPUT_DIR.resolve()),
    "steps": [
        "normalize_text",
        "convert_numeric",
        "replace_infinite_and_drop_missing_rows",
        "drop_exact_duplicates",
        "validate_ranges",
        "drop_difficulty_and_constant_columns",
        "add_attack_family_and_is_attack",
    ],
    "metadata_columns_removed": ["difficulty"],
    "constant_columns_removed": ["num_outbound_cmds"],
    "categorical_columns_preserved": categorical_cols,
    "target_columns_added": ["attack_family", "is_attack"],
}
with (OUTPUT_DIR / "cleaning_config.json").open("w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)

display(report_df)

## 12. Export summary

In [ ]:
print("Generated files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {path.name}: {path.stat().st_size:,} bytes")

print("\nCleaned training attack-family distribution:")
display(
    cleaned["KDDTrain"]["attack_family"]
    .value_counts()
    .rename_axis("attack_family")
    .reset_index(name="rows")
    .assign(percent=lambda x: x["rows"] / len(cleaned["KDDTrain"]) * 100)
)